# Network Defense RLReinforcement learning for sequential cyber defense, benchmarked against aQuantal Stackelberg Equilibrium solver.This notebook reproduces the full study. A defender allocates scarce hardeningand incident response effort across a network of varying size while aboundedly rational attacker probes it.The headline finding is a negative result and its fix. Learned agents matchedthe equilibrium solver on network sizes they trained on, then collapsed onlarger networks they had never seen. One diagnostic measurement explains why,and domain randomisation recovers almost all of the lost performance.Runtime is roughly 15 minutes on a Colab CPU instance. No GPU is needed.

In [ ]:
!pip install -q gymnasium stable-baselines3import numpy as npimport pandas as pdimport matplotlib.pyplot as pltnp.random.seed(0)print("ready")

## 1. The environmentEach episode draws a network of between 8 and 20 nodes. Every node has a valueand a defense level. Each step the defender acts on one node and the attackerattacks one node.Four design decisions matter.**Variable node count with padding.** Observations and actions pad to a fixed20 nodes with an active mask marking which slots are real, so tensor shapesstay constant while the underlying problem size varies. This is what makes thegeneralisation question askable at all.**A binding budget.** Defense decays every step. Without decay a defender with50 steps could max out every node at low node counts and allocation would stopbeing a decision. Sustainable defense mass is about 1 over the decay rate.**Prevention versus remediation.** Acting on a clean node hardens it. Actingon a compromised node cleans it. That tension is the interesting part of thepolicy.**Dense damage.** Compromised nodes bleed value every step rather thanincurring a single penalty, which gives a dense learning signal.

In [ ]:
# In Colab, clone the repo. Locally, the modules are already on the path.import osif not os.path.exists("network_defense_env.py"):    !git clone -q https://github.com/YOUR_USERNAME/network-defense-rl.git    %cd network-defense-rlfrom network_defense_env import NetworkDefenseEnvenv = NetworkDefenseEnv(seed=0)obs, info = env.reset(seed=0)print("observation shape :", obs.shape)print("action space      :", env.action_space)print("active nodes       :", info["n_active_nodes"])

### Checking the budget is actually bindingThis is worth verifying rather than assuming. If the defender can sustainenough coverage to protect every node, there is no allocation problem and thewhole study is measuring nothing.

In [ ]:
mass = env.sustainable_defense_mass()budget = mass / env.max_defense_levelprint(f"sustainable defense mass : {mass:.2f} levels")print(f"coverage budget          : {budget:.2f} nodes worth")print(f"nodes in this episode    : {info['n_active_nodes']}")print(f"binding                  : {budget < info['n_active_nodes']}")

## 2. BaselinesFour non learned policies. Random and value proportional are sanity floors.Greedy value is the heuristic a practitioner would actually reach for, sobeating it is the minimum bar.The fourth is the interesting one. QSE solves the one shot Stackelbergsecurity game underneath each step. The defender commits to a coverage vectorand a boundedly rational attacker quantal responds,    q_i(c) = softmax( lambda * v_i * (1 - c_i) )and the defender maximises    U_d(c) = - sum_i q_i(c) * v_i * (1 - c_i)subject to a budget constraint. The objective is not concave in general, so wesolve with SLSQP from several restarts.

In [ ]:
from baselines import solve_qse_, info = env.reset(seed=0, options={"n_nodes": 14})n = info["n_active_nodes"]values = env.value[:n]budget = env.sustainable_defense_mass() / env.max_defense_levelcoverage = solve_qse(values, budget, rationality=env.attacker_rationality)order = np.argsort(-values)fig, ax = plt.subplots(figsize=(9, 3.6))x = np.arange(n)ax.bar(x - 0.2, values[order], 0.4, label="node value", color="#b0b0b0")ax.bar(x + 0.2, coverage[order], 0.4, label="QSE coverage", color="#1b6ca8")ax.set_xlabel("nodes, sorted by value")ax.legend(frameon=False)ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()print(f"budget {budget:.2f}, allocated {coverage.sum():.2f}")print(f"nodes given zero coverage: {(coverage < 0.01).sum()} of {n}")

**Findings.** The solver concentrates coverage on high value nodes, abandonsthe cheapest ones entirely, and exactly saturates the budget. That is thestructure a Stackelberg security game should produce, which is a useful checkthat the solver is doing equilibrium reasoning rather than value weighting.A note on how this baseline was built. An earlier version of the sequentialQSE policy used an ad hoc remediation rule that compared quantities withmismatched units, and it made QSE lose to plain greedy. That was a bug in thebaseline, not a finding about equilibrium reasoning. Replacing it with aproper marginal utility comparison moved QSE to the top. A quietly crippledgame theoretic baseline would have made the learned agents look better thanthey are.

In [ ]:
from baselines import get_baselinesfrom evaluate import evaluate_policies, summarisebaseline_df = evaluate_policies(get_baselines(seed=0), n_episodes=100)summarise(baseline_df).round(4)

## 3. Training PPO and DQNBoth agents train only on networks of 8 to 14 nodes. The padded observationstays 20 wide, so the same policy can later be run on larger networks withoutany change.That setup is the experiment. If padding plus masking gives sizegeneralisation, an agent trained on small networks should handle large ones.

In [ ]:
from train import train_ppo, train_dqnppo_model, ppo_curve = train_ppo(total_timesteps=700_000, tag="ppo")dqn_model, dqn_curve = train_dqn(total_timesteps=150_000)curves = pd.concat([ppo_curve, dqn_curve], ignore_index=True)fig, ax = plt.subplots(figsize=(9, 3.8))for algo, g in curves.groupby("algo"):    g = g.sort_values("timestep")    ax.plot(g["timestep"], g["return"].rolling(150, min_periods=20).mean(), label=algo)ax.set_xlabel("environment steps"); ax.set_ylabel("episode return")ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()

## 4. The generalisation testEvery policy is evaluated twice. Once on 8 to 14 nodes, the range the agentstrained on, and once on 15 to 20 nodes, which they have never seen.All policies face identical episode seeds and identical node counts, sodifferences in return come from the policy rather than from luck in the draw.

In [ ]:
from run_evaluation import main as run_full_evaluationall_df, summary = run_full_evaluation()summary.round(4)

**Findings.** In distribution the learned agents are competitive. PPO reaches-2.91 against QSE at -2.75, and both beat the greedy heuristic.On held out network sizes PPO drops to -12.13 and DQN to -11.28, roughly fourtimes the value lost. QSE moves from -2.75 to -3.27 and greedy from -3.51 to-3.75, both essentially flat.The policies that never learned anything generalise. The ones that learned donot. That is the opposite of the intended result, so the next step is to findout why rather than to report it and move on.

## 5. Diagnosing the failureThe hypothesis is specific and testable. Agents trained on 8 to 14 nodes neversaw action indices 14 through 19 as valid. On held out networks those indicescover roughly a third of the nodes.If that is the cause, the trained policy should almost never select them. Ifgeneralisation were working, they should receive 30 to 40 percent of actions.

In [ ]:
from stable_baselines3 import PPOdef action_usage(model, n_episodes=150, node_range=(15, 20)):    e = NetworkDefenseEnv()    counts = np.zeros(20)    for ep in range(n_episodes):        n = int(np.random.default_rng(ep).integers(node_range[0], node_range[1] + 1))        obs, _ = e.reset(seed=10_000 + ep, options={"n_nodes": n})        for _ in range(50):            a, _ = model.predict(obs, deterministic=True)            counts[int(a)] += 1            obs, r, term, trunc, _ = e.step(a)            if trunc:                break    return countscounts = action_usage(PPO.load("models/ppo_defender"))print(f"actions on slots 0 to 13  : {int(counts[:14].sum())}")print(f"actions on slots 14 to 19 : {int(counts[14:].sum())}")print(f"share on never trained    : {counts[14:].sum() / counts.sum():.4f}")

**Findings.** PPO placed 6 actions out of roughly 7500 on the extra nodes, ashare of 0.08 percent against an expected 30 to 40 percent.It is not defending those nodes badly. It is not defending them at all. Theattacker takes a third of the network for free, which fully accounts for thesize of the collapse.The general lesson is that padding plus masking makes a variable sized problem*representable*, not *learnable*. If the training distribution never activatesa region of the action space, the policy has no gradient signal telling itanything about that region.

## 6. The fixIf the cause is that certain action slots were never active during training,the fix is to make them active. Domain randomisation over the full 8 to 20node range, changing nothing else about the architecture or hyperparameters.

In [ ]:
dr_model, dr_curve = train_ppo(    total_timesteps=700_000, node_range=(8, 20), tag="ppo_dr")dr_counts = action_usage(dr_model)print(f"PPO-DR share on slots 14 to 19: {dr_counts[14:].sum() / dr_counts.sum():.4f}")

In [ ]:
all_df, summary = run_full_evaluation()summary[summary.regime == "holdout"].sort_values("mean_return", ascending=False).round(4)

**Findings.** PPO-DR uses the previously dead action slots 27.7 percent of thetime against 0.08 percent for PPO, which confirms the mechanism directlyrather than by inference from the return alone.Held out return improves from -12.13 to -3.64. PPO-DR now beats the greedyheuristic on unseen sizes, and it costs about 0.15 return in distributionrelative to plain PPO, which is a small price for the robustness.QSE remains the strongest policy in both regimes.

## 7. Study 2, does learning ever beat the equilibrium?QSE needs the attacker's rationality parameter lambda as an input. An RLdefender never sees lambda at all, only reward. So there should be acrossover. When lambda is known, solving the equilibrium exactly should win.Once the assumed lambda is far enough from the truth, a policy learned acrossa range of attackers should win instead.Two arms make this testable. QSE that always assumes lambda equals 4 whateverthe true attacker is, and a PPO defender trained with lambda resampled loguniformly every episode so it cannot depend on knowing the attacker.

In [ ]:
from train import train_ppo# A defender that never learns a single attacker model.lam_model, _ = train_ppo(    total_timesteps=700_000,    node_range=(8, 20),    tag="ppo_lam",    env_kwargs={"rationality_range": (0.5, 16.0)},)

In [ ]:
from misspecification import main as run_misspecificationmiss_df, miss_summary = run_misspecification()

**Findings.** There is no crossover. QSE assuming lambda equals 4 never losesto any learned agent at any true lambda from 0.5 to 32, including the agenttrained across randomised attackers.A structural misspecification does not change this either. Swapping themultiplicative quantal response for an additive subjective utility form in thespirit of Nguyen et al. 2013 leaves QSE ahead at -2.96 against -3.36 for a PPOagent trained directly against that attacker.The next cell shows why.

In [ ]:
from baselines import solve_qseenv = NetworkDefenseEnv(seed=0)_, info = env.reset(seed=0, options={"n_nodes": 14})n = info["n_active_nodes"]v = env.value[:n]B = env.sustainable_defense_mass() / env.max_defense_levelbase = solve_qse(v, B, rationality=4.0)for lam in [0.5, 1, 2, 4, 8, 16, 32]:    c = solve_qse(v, B, rationality=lam)    print(f"lambda={lam:>4g}  correlation with lambda=4: {np.corrcoef(c, base)[0, 1]:.4f}")

**Findings.** The equilibrium coverage vector is almost invariant to lambda,correlation 0.92 to 1.00 across the whole range. Getting the attacker modelwrong barely changes what the defender should do, so there is nothing for alearner to exploit. Misspecification is not a lever in this environment.One result does come out of the sweep. Deliberately assuming a lower lambdathan the truth beats solving with the true one, gaining 0.055 return at truelambda 16 with paired p of 0.0005 and 0.061 at lambda 32 with p of 2e-05. Theone shot equilibrium over concentrates coverage for a sequential game withdecay and remediation, so a blunted attacker model produces a bettersequential policy. The optimal input to the solver is not the true parameter.

## 8. The ablation that reframes everythingIf the attacker model is not doing the work, what is? Two candidates. Eitherthe equilibrium allocation itself, or the policy structure wrapped around it,which tracks a coverage target and remediates on a marginal utility thresholdinstead of acting greedily.Replacing only the coverage target, with a naive fill that covers nodes indescending value order until the budget runs out, separates the two.

In [ ]:
from ablation import main as run_ablationabl_df, abl_summary = run_ablation()

**Findings.** The equilibrium solve beats naive fill by 0.042 return, pairedp = 0.05 over 300 shared seeds. Borderline at best.Almost all of QSE's advantage over the greedy heuristic comes from the policystructure and almost none from solving the game. QSE is also handed the exactattacker model the environment uses, an advantage the learned agents neverget.So the correct reading of the study 1 tables is not that equilibrium reasoningbeats reinforcement learning. It is that a good allocation-tracking heuristicbeats both, and in this environment the game theoretic machinery is close todecorative. Establishing that took an ablation the first version of this studydid not run, which is the general lesson worth carrying out of the notebook.

## LimitationsThe ablation above bounds what any QSE comparison here can claim. Equilibriumreasoning beats naive fill at p = 0.05, so treating game theory as the winningingredient would be overreach.The attacker is scripted rather than learned, so this is single agent RLagainst a fixed opponent and not true multi agent RL. Self play is the obviousextension.Training uses a single seed per agent. Evaluation uses shared episode seedswith reported standard errors, but small differences between learned agentsshould not be over read.DQN received 150k steps against PPO's 700k because single environment DQN withper step gradient updates is far slower in wall clock terms. Its numbers are areference point, not a converged ceiling.Node values are independent and uniform with no topology. That is also themost likely reason misspecification did not bite, since coupled nodes wouldmake the attacker model matter for the allocation in a way that independentnodes do not.